# Machine Learning Zoomcamp 2026 — Homework 1

A step-by-step learning notebook for [the official 2026 homework](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/01-intro/homework.md).
The questions below are paraphrased from that assignment.

Data: [official 2026 car fuel-efficiency CSV](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv).
The downloaded `car_fuel_efficiency_2026.csv` is beside this notebook. Run the cells from top to bottom in a Python environment with pandas and NumPy installed.
All results below are calculated from this file. Q1 reports the pandas version used to execute this notebook; your version may differ in another environment.

## Q1 — Installed pandas

**Question:** Which pandas release is running in this notebook?

In [ ]:
import pandas as pd
import numpy as np

pandas_version = pd.__version__
print("pandas version:", pandas_version)

pandas version: 3.0.6


`pd.__version__` reads the version string from the imported pandas package. `pd` and `np` are the usual short names for pandas and NumPy.

## Q2 — Dataset size

**Question:** What is the number of rows in the CSV?

In [2]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")
record_count = df.shape[0]
print("Dataset shape (rows, columns):", df.shape)
print("Number of records:", record_count)

Dataset shape (rows, columns): (10000, 11)
Number of records: 10000


`pd.read_csv()` loads the CSV into a DataFrame, a table with named columns. `df.shape` gives `(rows, columns)`, so `df.shape[0]` counts records; the header is not a record.

## Q3 — Fuel categories

**Question:** How many distinct fuel categories occur in the data?

In [3]:
fuel_type_count = df["fuel_type"].nunique()
print("Fuel categories:", df["fuel_type"].unique())
print("Number of fuel types:", fuel_type_count)

Fuel categories: <StringArray>
['Gasoline', 'Diesel', 'Hybrid']
Length: 3, dtype: str
Number of fuel types: 3


Selecting `df["fuel_type"]` gives one column as a Series. `unique()` shows its distinct values, and `nunique()` counts them. By default, `nunique()` excludes missing values.

## Q4 — Columns containing gaps

**Question:** How many columns contain at least one missing entry?

In [4]:
missing_counts = df.isna().sum()
columns_with_missing = (missing_counts > 0).sum()
print("Missing entries by column:")
print(missing_counts)
print("Number of columns with missing values:", columns_with_missing)

Missing entries by column:
model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64
Number of columns with missing values: 2


`isna()` marks missing entries with `True`. The first `sum()` counts them in each column. Comparing those counts with zero identifies columns containing gaps; the second `sum()` counts these columns because `True` counts as 1.

## Q5 — Best efficiency among Asian cars

**Question:** Among cars whose origin is Asia, what is the largest MPG value?

In [5]:
asia_cars = df[df["origin"] == "Asia"]
max_asia_efficiency = asia_cars["fuel_efficiency_mpg"].max()
print("Maximum fuel efficiency for Asia (MPG):", max_asia_efficiency)

Maximum fuel efficiency for Asia (MPG): 41.2


`df["origin"] == "Asia"` creates a Boolean mask. Indexing `df` with it keeps only Asian cars. `max()` finds the largest value in their fuel-efficiency column.

## Q6 — Horsepower before and after filling gaps

**Question:** Calculate the horsepower median and mode. Replace missing horsepower with the mode, then recalculate the median. Does it rise, fall, or stay the same?

In [6]:
median_before = df["horsepower"].median()
horsepower_modes = df["horsepower"].mode()
horsepower_mode = horsepower_modes.iloc[0]
horsepower_filled = df["horsepower"].fillna(horsepower_mode)
median_after = horsepower_filled.median()

if median_after > median_before:
    median_change = "increased"
elif median_after < median_before:
    median_change = "decreased"
else:
    median_change = "unchanged"

print("Median before filling:", median_before)
print("Most frequent horsepower value(s):", horsepower_modes.tolist())
print("Value used to fill gaps:", horsepower_mode)
print("Missing horsepower entries after filling:", horsepower_filled.isna().sum())
print("Median after filling:", median_after)
print("Median change:", median_change)

Median before filling: 254.0
Most frequent horsepower value(s): [252.0]
Value used to fill gaps: 252.0
Missing horsepower entries after filling: 0
Median after filling: 252.0
Median change: decreased


`median()` finds the middle value and ignores missing entries. `mode()` returns the most frequent value(s); `.iloc[0]` selects the first. `fillna()` returns a Series with gaps filled using that value. We keep it separately so the original DataFrame stays available for later questions.

## Q7 — Computing regression weights

**Question:** Take the first seven Asian cars in file order and retain `vehicle_weight` and `model_year`, in that order, as NumPy matrix `X`. Form `X.T @ X`, invert it, and compute `w = inv(X.T @ X) @ X.T @ y` for `y = [1100, 1300, 800, 900, 1000, 1100, 1200]`. What is the total of the weights?

In [7]:
X = asia_cars[["vehicle_weight", "model_year"]].head(7).to_numpy()
print("X shape:", X.shape)
print("X (vehicle_weight, model_year):")
print(X)

print("X.T shape:", X.T.shape)
print("X.T:")
print(X.T)

X shape: (7, 2)
X (vehicle_weight, model_year):
[[4240 1996]
 [4310 1992]
 [4230 1997]
 [4420 1978]
 [3950 2007]
 [4750 2021]
 [4450 2019]]
X.T shape: (2, 7)
X.T:
[[4240 4310 4230 4420 3950 4750 4450]
 [1996 1992 1997 1978 2007 2021 2019]]


Double brackets select two columns in the specified order. `head(7)` takes the first seven rows without sorting. `to_numpy()` extracts the numeric array: seven observations and two features give `X` shape `(7, 2)`. `.T` swaps rows and columns, giving `(2, 7)`.

In [8]:
XTX = X.T @ X
print("XTX shape:", XTX.shape)
print("XTX:")
print(XTX)

XTX_inverse = np.linalg.inv(XTX)
print("Inverse of XTX shape:", XTX_inverse.shape)
print("Inverse of XTX:")
print(XTX_inverse)

XTX shape: (2, 2)
XTX:
[[131950500  60750580]
 [ 60750580  28041424]]
Inverse of XTX shape: (2, 2)
Inverse of XTX:
[[ 2.96830537e-06 -6.43071025e-06]
 [-6.43071025e-06  1.39675281e-05]]


`@` performs matrix multiplication. Multiplying `(2, 7)` by `(7, 2)` produces a `(2, 2)` matrix. `np.linalg.inv()` computes its inverse, also `(2, 2)`; an inverse exists here because this matrix is nonsingular.

In [9]:
y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])
print("y shape:", y.shape)

inverse_times_transpose = XTX_inverse @ X.T
print("Inverse of XTX @ X.T shape:", inverse_times_transpose.shape)

w = inverse_times_transpose @ y
weights_sum = w.sum()
print("w shape:", w.shape)
print("w (vehicle_weight coefficient, model_year coefficient):")
print(w)
print("Sum of weights:", weights_sum)
print("Sum rounded to 3 decimal places:", round(weights_sum, 3))

y shape: (7,)
Inverse of XTX @ X.T shape: (2, 7)
w shape: (2,)
w (vehicle_weight coefficient, model_year coefficient):
[0.13644777 0.2327492 ]
Sum of weights: 0.36919696904925486
Sum rounded to 3 decimal places: 0.369


Multiplying `(2, 2)` by `(2, 7)` gives `(2, 7)`. Multiplying that result by the seven-element vector `y`, shape `(7,)`, gives two coefficients, shape `(2,)`. `w.sum()` adds them; rounding happens only for display.

**Why this is linear regression:** The prediction vector is `X @ w`. Least squares chooses `w` to minimize the sum of squared differences between these predictions and `y`. Setting the derivatives to zero gives the normal equations, `(X.T @ X) @ w = X.T @ y`. When `X.T @ X` is invertible, solving them gives `inv(X.T @ X) @ X.T @ y`.

Here each of the two selected features gets one coefficient. There is no intercept because we did not add a column of ones. The target `y` is the supplied exercise vector, not the dataset's MPG column.

## Calculated summary

The following cell reuses the results calculated above.

In [10]:
print(f"Q1 -> pandas {pandas_version}")
print(f"Q2 -> {record_count} records")
print(f"Q3 -> {fuel_type_count} fuel types")
print(f"Q4 -> {columns_with_missing} columns with missing values")
print(f"Q5 -> {max_asia_efficiency} MPG")
print(f"Q6 -> median {median_before} -> {median_after}; {median_change} (fill value: {horsepower_mode})")
print(f"Q7 -> sum of weights = {weights_sum:.12f} (rounded: {weights_sum:.3f})")

Q1 -> pandas 3.0.6
Q2 -> 10000 records
Q3 -> 3 fuel types
Q4 -> 2 columns with missing values
Q5 -> 41.2 MPG
Q6 -> median 254.0 -> 252.0; decreased (fill value: 252.0)
Q7 -> sum of weights = 0.369196969049 (rounded: 0.369)
